# 11 · Recommendation and ranking

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Compute item similarities, exclude seen items, and evaluate top-k recommendations.


## Step 1 — Create implicit feedback

A 1 means an observed interaction, not a guaranteed preference. Missing interactions are unknown.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
R=np.array([[1,1,0,0,0],[1,1,1,0,0],[0,0,1,1,0],[0,0,1,1,1],[1,0,0,0,1]],dtype=float)
similarity=cosine_similarity(R.T)
np.fill_diagonal(similarity,0)
print('Item similarity shape:',similarity.shape)


**Check your result:** Similarity is a 5 by 5 item matrix.


## Step 2 — Rank unseen items

Exclude interactions already observed for the current user.


In [ ]:
def recommend(user,k=2):
    scores=R[user] @ similarity
    scores[R[user] > 0]=-np.inf
    return np.argsort(-scores)[:min(k,int((R[user]==0).sum()))]
items=recommend(0)
assert not R[0,items].any()
print('Suggested item ids:',items)


**Check your result:** Only unseen items are returned.


## Step 3 — Measure hit rate without leaking the held-out interaction

Remove the held-out item before recomputing similarities. This tiny single-user exercise illustrates the protocol, not a reliable benchmark.


In [ ]:
held_out=1
train=R.copy()
train[0,held_out]=0
sim=cosine_similarity(train.T)
np.fill_diagonal(sim,0)
scores=train[0] @ sim
scores[train[0] > 0]=-np.inf
top2=np.argsort(-scores)[:2]
print('Hit@2:',int(held_out in top2),'ranking:',top2)


**Check your result:** Hit@2 is either zero or one; held-out feedback was excluded.


## Practice

Evaluate Recall@k and NDCG@k across many users with chronological holdouts. Add a popularity baseline.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Keep training interactions before the cutoff, remove seen items, and aggregate per-user metrics. Report cold-start performance separately.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
